In [0]:
import requests
import pandas as pd

def fetch_imf_data(database, frequency, countries, indicator, start_year, end_year):
    # 1. ประกอบ URL
    base_url = "http://dataservices.imf.org/REST/SDMX_JSON.svc/CompactData"
    url = f"{base_url}/{database}/{frequency}.{countries}.{indicator}?startPeriod={start_year}&endPeriod={end_year}"
    
    print(f"Fetching data from: {url}")
    
    # 2. ยิง Request (IMF API บังคับให้ส่ง Accept header แบบนี้)
    headers = {'Accept': 'application/json'}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print("Error fetching data")
        return None
        
    data = response.json()
    
    # 3. แกะกล่อง JSON SDMX 
    try:
        series_list = data['CompactData']['DataSet']['Series']
    except KeyError:
        print("ไม่พบข้อมูล หรือพารามิเตอร์ผิดพลาด")
        return None

    # ถ้าดึงประเทศเดียว ข้อมูลจะเป็น dict ต้องครอบด้วย list ให้วนลูปได้
    if isinstance(series_list, dict):
        series_list = [series_list]

    # 4. ดึงข้อมูลลง List เพื่อทำ DataFrame
    records = []
    for series in series_list:
        country = series.get('@REF_AREA')
        
        obs_data = series.get('Obs', [])
        if isinstance(obs_data, dict):
            obs_data = [obs_data]
            
        for obs in obs_data:
            records.append({
                'Country': country,
                'Indicator': indicator,
                'Year': obs.get('@TIME_PERIOD'),
                'Value': float(obs.get('@OBS_VALUE', 0))
            })

    # แปลงเป็น Pandas DataFrame
    df = pd.DataFrame(records)
    return df




In [0]:
df_imf = fetch_imf_data(
        database="IFS", 
        frequency="A", 
        countries="TH+US+JP", 
        indicator="PCPI_IX", 
        start_year="2003", 
        end_year="2023"
)
    
print("\n--- ผลลัพธ์ข้อมูล (5 แถวแรก) ---")
print(df_imf.head())